# 🔍 모델 특징 시각화 (Model Feature Visualization)

이 노트북에서는 학습된 CNN/CRNN 모델이 어떤 특징을 학습했는지 시각화합니다.

## 📋 시각화 항목
1. **CNN 특징 맵 (Feature Maps)**: 각 Conv 레이어가 감지하는 패턴
2. **CNN 필터 가중치**: 학습된 필터의 모양
3. **CRNN Attention Weights**: 모델이 집중하는 시간 구간
4. **Grad-CAM**: 분류에 중요한 영역 시각화

In [1]:
# ============================================================
# 필수 라이브러리 임포트
# ============================================================

import os
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

# 공통 유틸리티
from utils import setup_plotting, get_data_dir, get_state_mapping, get_state_names

# 프로젝트 모듈
from app.ml.features.extractor import AudioFeatureExtractor, AudioConfig
from app.ml.models.cnn import SoundClassifierCNN
from app.ml.models.crnn import SoundClassifierCRNN

# 시각화 설정
setup_plotting()

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 라이브러리 로드 완료!")
print(f"🖥️ Device: {device}")


✅ 라이브러리 로드 완료!
🖥️ Device: cpu


---
## 1. CNN 특징 맵 시각화 함수


In [2]:
# ============================================================
# CNN 중간 레이어 특징 맵 추출 및 시각화
# ============================================================

def extract_cnn_feature_maps(model, x, layer_names=['conv1', 'conv2', 'conv3', 'conv4']):
    """CNN 모델의 중간 레이어 특징 맵 추출"""
    model.eval()
    feature_maps = {}
    
    with torch.no_grad():
        current = x
        
        if 'conv1' in layer_names:
            conv1_out = model.conv1(current)
            feature_maps['conv1'] = conv1_out.cpu().numpy()
            current = conv1_out
        
        if 'conv2' in layer_names:
            conv2_out = model.conv2(current)
            feature_maps['conv2'] = conv2_out.cpu().numpy()
            current = conv2_out
        
        if 'conv3' in layer_names:
            conv3_out = model.conv3(current)
            feature_maps['conv3'] = conv3_out.cpu().numpy()
            current = conv3_out
        
        if 'conv4' in layer_names:
            conv4_out = model.conv4(current)
            feature_maps['conv4'] = conv4_out.cpu().numpy()
    
    return feature_maps

def visualize_cnn_feature_maps(feature_maps, sample_name, layer_name, n_filters=16):
    """CNN 특징 맵 시각화"""
    fm = feature_maps[0]  # 첫 번째 배치
    n_channels = fm.shape[0]
    n_show = min(n_filters, n_channels)
    
    cols = 4
    rows = (n_show + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx in range(n_show):
        row = idx // cols
        col = idx % cols
        ax = axes[row, col]
        
        feature_map = fm[idx]
        im = ax.imshow(feature_map, aspect='auto', origin='lower', cmap='viridis')
        ax.set_title(f'Filter {idx}', fontsize=10)
        ax.set_xlabel('Time')
        ax.set_ylabel('Frequency')
        plt.colorbar(im, ax=ax)
    
    for idx in range(n_show, rows * cols):
        row = idx // cols
        col = idx % cols
        axes[row, col].axis('off')
    
    plt.suptitle(f'🔍 CNN {layer_name} 특징 맵 - {sample_name}', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

print("✅ CNN 특징 맵 시각화 함수 정의 완료!")


✅ CNN 특징 맵 시각화 함수 정의 완료!


---
## 2. CRNN Attention Weights 시각화 함수


In [3]:
# ============================================================
# CRNN Attention Weights 시각화
# ============================================================

def visualize_crnn_attention(model, sample_data, state_name, state_names):
    """CRNN 모델의 Attention weights 시각화"""
    model.eval()
    
    x = sample_data['tensor']
    mel_spec = sample_data['mel_spec'][0]  # (128, 216)
    
    with torch.no_grad():
        output, attention_weights = model.forward_with_attention(x)
        probs = F.softmax(output, dim=1)
        pred_class = output.argmax(dim=1).item()
        confidence = probs[0, pred_class].item()
    
    attn = attention_weights.cpu().numpy().squeeze()  # (time_frames,)
    
    # 시각화
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # 1. 원본 스펙트로그램
    ax = axes[0]
    im = ax.imshow(mel_spec, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(f'🎵 원본 멜 스펙트로그램 - {state_name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time (frames)')
    ax.set_ylabel('Frequency (Mel bands)')
    plt.colorbar(im, ax=ax, label='Magnitude (dB)')
    
    # 2. Attention Weights
    ax = axes[1]
    time_frames = np.arange(len(attn))
    ax.plot(time_frames, attn, 'b-', linewidth=2, label='Attention Weight')
    ax.fill_between(time_frames, 0, attn, alpha=0.3, color='blue')
    ax.set_xlabel('Time (frames)', fontsize=11)
    ax.set_ylabel('Attention Weight', fontsize=11)
    ax.set_title(f'🎯 Attention Weights (예측: {state_names[pred_class]}, 신뢰도: {confidence:.2%})', 
                 fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    max_idx = np.argmax(attn)
    ax.axvline(x=max_idx, color='red', linestyle='--', linewidth=2, 
               label=f'Peak: frame {max_idx}')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    return attn, pred_class, confidence

print("✅ CRNN Attention 시각화 함수 정의 완료!")


✅ CRNN Attention 시각화 함수 정의 완료!


---
## 3. 사용 예시

**사용 방법:**
1. 학습된 모델 체크포인트를 로드
2. 샘플 데이터 준비
3. 위의 시각화 함수들을 호출하여 특징 확인

**시각화 가능한 항목:**
- ✅ CNN 각 레이어의 특징 맵 (conv1, conv2, conv3, conv4)
- ✅ CNN 필터 가중치
- ✅ CRNN Attention Weights (어떤 시간 구간이 중요한지)
- ✅ Grad-CAM (분류에 중요한 영역)

**참고:** 실제 사용 시에는 학습된 모델과 샘플 데이터가 필요합니다.
